# Part 5: SVI and using SVI to warm up NUTS
**Stochastic variational inference (SVI)** optimizes a variational family (e.g. diagonal Normal) to approximate the posterior; it is often **faster** than MCMC for a first approximation. You can also use SVI to **warm up NUTS**: run SVI, then start NUTS from the variational mean via `init_to_value`. That can reduce NUTS warmup time and help when default init is poor. SVI also helps you **rapidly prototype** Bayesian inference before committing to longer MCMC runs.

## 5.1 SVI on the filter-based model

We use the same stochastic volatility model with the **filter** (so we only infer $\phi$). SVI with `AutoDiagonalNormal` gives a Gaussian approximation to $p(\phi \mid \text{data})$.

In [2]:
import time
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
import optax
import dynestyx as dsx
from dynestyx import (
    Condition, 
    Context, 
    DynamicalModel, 
    DiscreteTimeSimulator, 
    FilterBasedMarginalLogLikelihood, 
    Trajectory
)
from numpyro.infer import MCMC, NUTS, Predictive, SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDiagonalNormal
from numpyro.infer.initialization import init_to_value

def stochastic_volatility_model():
    phi = numpyro.sample("phi", dist.Uniform(0.0, 1.0))
    sigma_eta = 0.5
    initial_condition = dist.Normal(0.0, 1.0)
    def state_evolution(x, u, t_now, t_next):
        return dist.Normal(phi * x, sigma_eta)
    def observation_model(x, u, t):
        return dist.Normal(0.0, jnp.exp(x / 2.0))
    dynamics = DynamicalModel(
        state_dim=1, observation_dim=1, control_dim=0,
        initial_condition=initial_condition,
        state_evolution=state_evolution,
        observation_model=observation_model,
    )
    return dsx.sample("f", dynamics)

key = jr.PRNGKey(0)
phi_true = 0.9
obs_times = jnp.arange(0.0, 100.0, 1.0)
context_gen = Context(observations=Trajectory(times=obs_times), controls=Trajectory())
predictive = Predictive(stochastic_volatility_model, params={"phi": jnp.array(phi_true)}, num_samples=1, exclude_deterministic=False)
with DiscreteTimeSimulator():
    with Condition(context_gen):
        synthetic = predictive(key)
obs_values = synthetic["observations"][0]
observation_trajectory = Trajectory(times=obs_times, values=obs_values)

def filter_conditioned_model():
    context = Context(observations=observation_trajectory, controls=Trajectory())
    with FilterBasedMarginalLogLikelihood(filter_type="pf", n_filter_particles=1000):
        with Condition(context):
            return stochastic_volatility_model()

guide = AutoDiagonalNormal(filter_conditioned_model)
optimizer = optax.adam(learning_rate=1e-3)
svi = SVI(filter_conditioned_model, guide, optimizer, loss=Trace_ELBO())

t0 = time.perf_counter()
svi_result = svi.run(jr.PRNGKey(1), num_steps=300)
svi_time = time.perf_counter() - t0
print(f"SVI (300 steps): {svi_time:.2f} s")
svi_params = svi_result.params
phi_svi_mean = guide.median(svi_params)["phi"]
print("SVI median(phi):", float(phi_svi_mean))

ValueError: not enough values to unpack (expected 3, got 2)

## 5.2 NUTS from SVI initialization

We run NUTS with `init_to_value` set to the SVI median (or mean). This can reduce warmup time and improve mixing when the prior is diffuse.

In [ ]:
init_values = guide.median(svi_params)

nuts_kernel = NUTS(filter_conditioned_model, init_strategy=init_to_value(values=init_values))
mcmc = MCMC(nuts_kernel, num_warmup=150, num_samples=150)
t0 = time.perf_counter()
mcmc.run(jr.PRNGKey(2))
nuts_time = time.perf_counter() - t0
print(f"NUTS (150 warmup + 150 samples): {nuts_time:.2f} s")

posterior = mcmc.get_samples()
print("NUTS posterior phi mean:", float(jnp.mean(posterior["phi"])))
print("True phi:", phi_true)

On harder models (longer trajectories, more parameters), NUTS warmup can be slow; SVI gives a cheap initialization and a fast way to prototype before running full MCMC.

**Next:** [Part 6 — Continuous-time dynamical systems](../06_continuous_time.ipynb)